In [ ]:
import ast
import datetime
import json
import numpy as np
import pandas as pd
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
import spacy
from spacy.tokens import Span, Doc

from dap_job_quality.utils.keyword_search_patterns import keywords
from dap_job_quality.getters.ojo_getters import get_ojo_sample
from dap_job_quality.utils.spacy_keyword_search import get_matches, get_spans
from dap_job_quality.utils.text_cleaning import clean_text

from dap_job_quality.getters.data_getters import load_s3_jsonl
from dap_job_quality.getters.labelled_data import get_labelled_job_sentences
from dap_job_quality.utils import prodigy_data_utils as pdu

from dap_job_quality import BUCKET_NAME, PROJECT_DIR, config

model = SentenceTransformer("all-MiniLM-L6-v2")

pd.set_option('display.width', 1000)

TODAY = datetime.datetime.today().strftime('%Y-%m-%d')

In [ ]:
nlp = spacy.load("en_core_web_sm")

SEED = config["seed"]

In [ ]:
labelled_data = pd.read_csv(PROJECT_DIR / "inputs/labelled/Mapping evaluation - v2.csv", index_col=0)
lookup = pd.read_csv(PROJECT_DIR / "inputs/keyword_lookup - v5.csv")

labelled_data['span'] = labelled_data['span'].apply(lambda x: ast.literal_eval(x))

In [ ]:
def count_words(text):
    return len(text.split())

lookup['word_count'] = lookup['target_phrase'].apply(count_words)

max_word_count = lookup['word_count'].max()

max_word_count


In [ ]:
labelled_spans = labelled_data.explode(['span'])

In [ ]:
labelled_data.iloc[1]['span']

In [ ]:
labelled_spans.iloc[1]['span']

In [ ]:
labelled_spans.iloc[2]['span']

In [ ]:
labelled_spans['span_length'] = labelled_spans['span'].apply(count_words)

In [ ]:
labelled_spans['span_length'].max()

In [ ]:
def generate_ngrams(text, n=4):
    words = text.split()
    ngrams = [' '.join(words[i:i+n]) for i in range(len(words) - n + 1)]
    return ngrams

def split_text(text):
    # Regular expression pattern to match ',', ';', or ':'
    pattern = r'[;,:]'
    # Use re.split() to split the text on the pattern
    return re.split(pattern, text)

def split_text_on_cc(text):
    # Process the text with spaCy
    doc = nlp(text)
    
    # Find the indices of tokens with the 'CC' dependency label
    cc_indices = [token.i for token in doc if token.dep_ == 'cc']
    
    # Initialize the start index and list for split segments
    start_idx = 0
    segments = []
    
    # Split the text at each 'CC' token
    for idx in cc_indices:
        segments.append(doc[start_idx:idx].text.strip())
        start_idx = idx + 1
    
    # Append the last segment
    segments.append(doc[start_idx:].text.strip())
    
    return segments

def split_text_on_phrases(text):
    # Regular expression pattern to match the phrases "and a" or "with a"
    pattern = r'with the|with a'
    # Use re.split() to split the text on the pattern
    return re.split(pattern, text)

In [ ]:
sample_sents = ['In return you will receive an attractive package, long term work opportunities and a clear path to progress to Project manager with a 6 month performance review.',
                'This role will be based in one of our offices (London, Cardiff, Edinburgh).',
                'We follow a hybrid working arrangement with a minimum of two days in the office.',
                'Join us to develop your strengths and enjoy a fulfilling career full of varied experiences',
                'The organisation also gets the global team together once in the summer and once for the Christmas party - usually an international trip to say thank you.',
                'The role is 36 hours per week, working 5 days out of 7.',
                'Free onsite parking and access to electric charging points.',
                'My client is offering £25,000 - £30,000 plus benefits depending on experience with the ability to work from home 2 3 days a week.',
                'They offer an ego and politics free working environment and subsequently enjoy a high retention rate',
                'Professional and personal learning and development opportunities.',
                'The opportunity to work for a leading international corporation']

In [ ]:
for sent in sample_sents:
    print(split_text_on_cc(sent))
    print()

In [ ]:
doc = nlp('We follow a hybrid working arrangement with a minimum of two days in the office.')
for token in doc:
    print(token, token.dep_)

In [ ]:
phrases = split_text(sentence)
phrases

In [ ]:
phrases[1].split('and a')

In [ ]:
generate_ngrams(sentence)